# Task B -- five-seed full fits

Six ideas, five seeds each, every model trained on **all 3,143 rows**. No folds,
no held-out split. Five models per arm, probabilities averaged.

Roughly 9.5 hours: 25 minutes of domain adaptation, then 1 hour 35 per arm.
The budget guard drops arms from the bottom of the list if time runs short, so
this cannot be cut off mid-run.

**No arm here has a score, and none can.** Every labelled row is in training.
Rank on the `holdout` column, which is what each idea already scored on the
fixed 472-row split from last night. The two label-free diagnostics are
described in `RESULTS.md`.

Dropped as measurably bad: XLM-R at 0.5561 and mDeBERTa at 0.1002.

Sidebar: Accelerator `GPU T4 x2` or `GPU P100`, Internet on. Save & Run All.


In [ ]:
import os, subprocess, sys, pathlib
WORK="/kaggle/working/hastika"
if os.path.isdir(WORK+"/.git"):
    subprocess.run(["git","-C",WORK,"pull","--ff-only"], check=True)
else:
    subprocess.run(["git","clone","-q","-b","task-b","--depth","1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git",WORK], check=True)
os.chdir(WORK); print("cwd:",os.getcwd())
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))


## Smoke test

Two minutes, fails loudly. Confirms the GPU path before nine hours of it.


In [ ]:
subprocess.run([sys.executable,"-u","work/muril_b.py","--tag","smoke",
                "--folds","1","--epochs","1","--limit","200","--seeds","42"], check=True)


## The sweep


In [ ]:
cmd=[sys.executable,"-u","work/fullfit_b.py",
     "--budget-hours","10.5","--reserve-min","15",
     "--seeds","42 43 44 45 46","--epochs","6","--out","/kaggle/working"]
print(" ".join(cmd), flush=True)
p=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout: sys.stdout.write(line)
p.wait(); print("sweep exit", p.returncode)


## What to submit


In [ ]:
print(pathlib.Path("/kaggle/working/RESULTS.md").read_text())
print("\nzips:")
for z in sorted(pathlib.Path("/kaggle/working/subs").glob("*.zip")):
    print(" ", z.name, f"{z.stat().st_size/1024:.1f} KB")


## Reading the table

Rank on `holdout`. It is the only column carrying evidence about quality, and
`f_tapt` leads it at 0.5972.

Use `drift` as a veto, not a ranking. It is the largest gap in points between an
arm's predicted class rates and the training prior. Around 3 is normal. A large
number means the model skewed toward a class; mDeBERTa's collapse would have
shown as 57 here.

Use `agree` for expectations. It is how often an arm matches the submission that
scored 0.5922. At 95% expect a similar score. At 70% it is a genuinely different
bet, which could go either way.
